# 01 — Data NHANES: unduh dan bangun dataset

**Artikel:** Fairness model deteksi diabetes belum terdiagnosis (NHANES) — target *Computers in Biology and Medicine*.

Notebook ini:
1. Mengunduh file NHANES siklus 2011–2012, 2013–2014, 2015–2016, dan 2017–Maret 2020 ke `data\raw`.
2. Menggabungkan semua siklus, menerapkan kriteria inklusi, membentuk outcome (HbA1c ≥ 6,5%), dan menghitung bobot survei gabungan.
3. Menyimpan dataset analisis dan tabel alur sampel.

Jalankan sel dari atas ke bawah (**Shift + Enter**). Unduhan hanya perlu sekali; file yang sudah ada dilewati.

## 0. Persiapan

Hapus tanda `#` pada baris `%pip` di bawah **hanya saat pertama kali** untuk memasang paket, lalu pasang lagi tandanya.

In [ ]:
# %pip install pandas numpy scikit-learn xgboost lightgbm fairlearn shap matplotlib samplics

In [ ]:
from pathlib import Path
import time
import urllib.request

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent  # repository root (run notebooks from notebooks/)
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
HASIL = ROOT / "hasil"
for folder in (RAW, PROCESSED, HASIL):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
print("Folder kerja:", ROOT)

## 1. Unduh data NHANES dari CDC

Total 36 file `.xpt` (4 siklus × 9 komponen). Siklus 2017–2020 memakai tekanan darah osilometrik (`BPXO`).
Kalau ada file yang gagal, unduh manual dari situs NHANES dan simpan di `data\raw` dengan nama yang sama.

In [ ]:
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt"

# siklus -> (tahun awal untuk URL, cara memberi nama file, kolom bobot MEC, lama siklus dalam tahun)
CYCLES = {
    "2011-2012": ("2011", lambda c: f"{c}_G", "WTMEC2YR", 2.0),
    "2013-2014": ("2013", lambda c: f"{c}_H", "WTMEC2YR", 2.0),
    "2015-2016": ("2015", lambda c: f"{c}_I", "WTMEC2YR", 2.0),
    "2017-2020": ("2017", lambda c: f"P_{c}", "WTMECPRP", 3.2),
}
KOMPONEN = ["DEMO", "DIQ", "GHB", "BMX", "BPQ", "MCQ", "SMQ", "PAQ"]


def komponen_tekanan_darah(siklus):
    return "BPXO" if siklus == "2017-2020" else "BPX"


def unduh(url, tujuan, percobaan=3):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (research download)"})
    for i in range(1, percobaan + 1):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                isi = resp.read()
            if not isi.startswith(b"HEADER RECORD"):
                raise ValueError("isi bukan file XPT (kemungkinan URL salah atau halaman error)")
            tujuan.write_bytes(isi)
            return True
        except Exception as e:
            print(f"   percobaan {i} gagal: {e}")
            time.sleep(3 * i)
    return False


gagal = []
for siklus, (tahun, namer, _, _) in CYCLES.items():
    for komp in KOMPONEN + [komponen_tekanan_darah(siklus)]:
        nama = namer(komp)
        tujuan = RAW / f"{nama}.xpt"
        if tujuan.exists() and tujuan.stat().st_size > 0:
            print(f"[ada]    {siklus} {nama}")
            continue
        url = BASE_URL.format(year=tahun, name=nama)
        print(f"[unduh]  {siklus} {nama}")
        if not unduh(url, tujuan):
            gagal.append(url)

print()
print("Gagal diunduh:", gagal if gagal else "tidak ada — semua file lengkap")

## 2. Gabungkan semua siklus

Komponen tiap siklus digabung berdasarkan `SEQN`. Bobot gabungan mengikuti panduan NHANES:
bobot MEC × 2/9,2 untuk siklus 2 tahun dan × 3,2/9,2 untuk 2017–Maret 2020.

In [ ]:
TOTAL_TAHUN = sum(v[3] for v in CYCLES.values())  # 9,2 tahun

KOLOM = {
    "DEMO": ["SEQN", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "INDFMPIR",
             "DMDEDUC2", "RIDEXPRG", "SDMVPSU", "SDMVSTRA"],
    "DIQ": ["SEQN", "DIQ010", "DIQ050", "DIQ070"],
    "GHB": ["SEQN", "LBXGH"],
    "BMX": ["SEQN", "BMXBMI", "BMXWAIST"],
    "BPQ": ["SEQN", "BPQ020"],
    "MCQ": ["SEQN", "MCQ300C"],
    "SMQ": ["SEQN", "SMQ020"],
    "PAQ": ["SEQN", "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665"],
}


def baca(nama_file):
    path = RAW / f"{nama_file}.xpt"
    if not path.exists():
        raise FileNotFoundError(f"File {path} belum ada. Jalankan bagian 1 (unduh) dulu.")
    return pd.read_sas(path, format="xport", encoding="utf-8")


def ambil(df, kolom):
    # kolom yang tidak ada di suatu siklus diisi NaN
    out = pd.DataFrame(index=df.index)
    for k in kolom:
        out[k] = df[k] if k in df.columns else np.nan
    return out


def tekanan_darah(namer, siklus):
    if siklus == "2017-2020":
        bp = baca(namer("BPXO"))
        sys_cols = [c for c in ["BPXOSY1", "BPXOSY2", "BPXOSY3"] if c in bp.columns]
        dia_cols = [c for c in ["BPXODI1", "BPXODI2", "BPXODI3"] if c in bp.columns]
    else:
        bp = baca(namer("BPX"))
        sys_cols = [c for c in ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"] if c in bp.columns]
        dia_cols = [c for c in ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"] if c in bp.columns]
    dia = bp[dia_cols].where(bp[dia_cols] > 0)  # diastolik 0 = tidak valid
    return pd.DataFrame({"SEQN": bp["SEQN"],
                         "SBP_MEAN": bp[sys_cols].mean(axis=1),
                         "DBP_MEAN": dia.mean(axis=1)})


def satu_siklus(siklus, namer, kolom_bobot, lama):
    demo = baca(namer("DEMO"))
    df = ambil(demo, KOLOM["DEMO"])
    df["WT_MEC_ASLI"] = demo[kolom_bobot]
    for komp in ["DIQ", "GHB", "BMX", "BPQ", "MCQ", "SMQ", "PAQ"]:
        df = df.merge(ambil(baca(namer(komp)), KOLOM[komp]), on="SEQN", how="left")
    df = df.merge(tekanan_darah(namer, siklus), on="SEQN", how="left")
    df["SIKLUS"] = siklus
    df["WT_GABUNGAN"] = df["WT_MEC_ASLI"] * lama / TOTAL_TAHUN
    return df


gabung = pd.concat(
    [satu_siklus(s, namer, bobot, lama) for s, (_, namer, bobot, lama) in CYCLES.items()],
    ignore_index=True,
)
print(f"Total peserta semua siklus: {len(gabung):,}")
gabung["SIKLUS"].value_counts().sort_index()

## 3. Kriteria inklusi dan alur sampel

Tabel alur ini dipakai untuk diagram alur peserta di naskah.

In [ ]:
HBA1C_CUTOFF = 6.5  # %, kriteria ADA

alur = [("Seluruh peserta NHANES 2011-Maret 2020", len(gabung))]


def saring(df, mask, keterangan):
    df = df[mask.fillna(False)]
    alur.append((keterangan, len(df)))
    return df


df = gabung
df = saring(df, df["RIDSTATR"] == 2, "Diperiksa di MEC")
df = saring(df, df["RIDAGEYR"] >= 20, "Usia >= 20 tahun")
df = saring(df, df["RIDEXPRG"] != 1, "Tidak hamil")
df = saring(df, df["DIQ010"].isin([2, 3]), "Belum pernah didiagnosis diabetes (DIQ010 = tidak/batas)")
df = saring(df, (df["DIQ050"] != 1) & (df["DIQ070"] != 1), "Tidak memakai insulin atau obat diabetes")
df = saring(df, df["LBXGH"].notna(), "Memiliki data HbA1c")
df = saring(df, df["WT_GABUNGAN"] > 0, "Bobot MEC > 0")

tabel_alur = pd.DataFrame(alur, columns=["Tahap", "N"])
tabel_alur.to_csv(HASIL / "alur_sampel.csv", index=False)
tabel_alur

## 4. Rekode variabel dan simpan dataset analisis

- **Outcome:** `DM_BELUM_TERDIAGNOSIS` = 1 jika HbA1c ≥ 6,5%.
- **Atribut sensitif:** kelompok usia, jenis kelamin, ras/etnis, kelompok pendapatan (rasio pendapatan/garis kemiskinan).
- **Prediktor non-laboratorium:** BMI, lingkar pinggang, tekanan darah, riwayat hipertensi, riwayat keluarga diabetes, merokok, aktivitas fisik, pendidikan.

In [ ]:
def kode_hilang(s, kode=(7, 9, 77, 99, 777, 999)):
    return s.where(~s.isin(kode))


def ya_tidak(s):
    # 1 = ya, 2 = tidak -> 1/0; kode lain -> NaN
    return s.map({1: 1.0, 2: 0.0})


data = pd.DataFrame({"SEQN": df["SEQN"].astype("int64"), "SIKLUS": df["SIKLUS"]})

# outcome
data["HBA1C"] = df["LBXGH"]
data["DM_BELUM_TERDIAGNOSIS"] = (df["LBXGH"] >= HBA1C_CUTOFF).astype(int)

# atribut sensitif
data["USIA"] = df["RIDAGEYR"]
data["KEL_USIA"] = pd.cut(df["RIDAGEYR"], bins=[19, 39, 59, 200], labels=["20-39", "40-59", "60+"])
data["JENIS_KELAMIN"] = df["RIAGENDR"].map({1: "Laki-laki", 2: "Perempuan"})
data["RAS_ETNIS"] = df["RIDRETH3"].map({1: "Mexican American", 2: "Other Hispanic", 3: "NH White",
                                        4: "NH Black", 6: "NH Asian", 7: "Other/Multiracial"})
data["PIR"] = df["INDFMPIR"]
data["KEL_PENDAPATAN"] = pd.cut(df["INDFMPIR"], bins=[-0.01, 1.3, 3.5, 10], right=False,
                                labels=["<1,3", "1,3-<3,5", ">=3,5"]).astype(object).fillna("Tidak diketahui")

# prediktor non-laboratorium
data["BMI"] = df["BMXBMI"]
data["LINGKAR_PINGGANG"] = df["BMXWAIST"]
data["SBP"] = df["SBP_MEAN"]
data["DBP"] = df["DBP_MEAN"]
data["RIWAYAT_HIPERTENSI"] = ya_tidak(df["BPQ020"])
data["RIWAYAT_KELUARGA_DM"] = ya_tidak(df["MCQ300C"])
data["PERNAH_MEROKOK"] = ya_tidak(df["SMQ020"])
aktivitas = pd.concat([ya_tidak(df[c]) for c in ["PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665"]], axis=1)
data["AKTIF_FISIK"] = aktivitas.max(axis=1)  # 1 jika ada aktivitas sedang/berat apa pun
data["PENDIDIKAN"] = kode_hilang(df["DMDEDUC2"])
data["PRADIABETES_DIBERITAHU"] = (df["DIQ010"] == 3).astype(int)

# desain survei
data["WT_GABUNGAN"] = df["WT_GABUNGAN"]
data["SDMVPSU"] = df["SDMVPSU"]
data["SDMVSTRA"] = df["SDMVSTRA"]

data.to_csv(PROCESSED / "nhanes_dm_belum_terdiagnosis.csv", index=False)
print(f"Dataset disimpan: {PROCESSED / 'nhanes_dm_belum_terdiagnosis.csv'}  ({len(data):,} baris)")
data.head()

## 5. Cek cepat: data hilang dan prevalensi per kelompok

Prevalensi di sini berbobot survei, tetapi belum memakai galat baku desain survei (itu di notebook berikutnya).

In [ ]:
data_hilang = (data.isna().mean() * 100).round(1).rename("% hilang")
data_hilang[data_hilang > 0]

In [ ]:
baris = []
for kolom in ["KEL_USIA", "JENIS_KELAMIN", "RAS_ETNIS", "KEL_PENDAPATAN", "SIKLUS"]:
    for nilai, g in data.groupby(kolom, observed=True):
        baris.append({
            "Atribut": kolom, "Kelompok": nilai, "N": len(g),
            "Kasus": int(g["DM_BELUM_TERDIAGNOSIS"].sum()),
            "Prevalensi berbobot (%)": round(100 * np.average(g["DM_BELUM_TERDIAGNOSIS"], weights=g["WT_GABUNGAN"]), 2),
        })
prevalensi = pd.DataFrame(baris)
prevalensi.to_csv(HASIL / "prevalensi_per_kelompok.csv", index=False)

total = np.average(data["DM_BELUM_TERDIAGNOSIS"], weights=data["WT_GABUNGAN"])
print(f"Kasus diabetes belum terdiagnosis: {int(data['DM_BELUM_TERDIAGNOSIS'].sum()):,} dari {len(data):,} "
      f"(prevalensi berbobot {100 * total:.2f}%)")
prevalensi

## Langkah berikutnya

Kirim ke Claude isi `hasil\alur_sampel.csv` dan `hasil\prevalensi_per_kelompok.csv` (atau tangkapan layar dua tabel di atas).
Setelah itu lanjut ke notebook **02 — model baseline dan audit fairness**.